# Búsqueda binaria

En una secuencia **ordenada**, compara con el elemento central y descarta la mitad donde no puede estar el objetivo. Cuesta $O(\log n)$.

## 1. Búsqueda binaria iterativa

Devolver el índice de un valor en una lista ordenada o −1 si no está.

In [1]:
def busqueda_binaria(lista, objetivo):
    izq, der = 0, len(lista) - 1
    while izq <= der:
        medio = (izq + der) // 2
        if lista[medio] == objetivo:
            return medio
        if lista[medio] < objetivo:
            izq = medio + 1
        else:
            der = medio - 1
    return -1

datos = [1, 3, 4, 7, 9, 11, 15]
print(busqueda_binaria(datos, 9), busqueda_binaria(datos, 8))

4 -1


## 2. Primera aparición

Con valores repetidos, devolver la primera posición donde podría insertarse el objetivo manteniendo el orden (como `bisect.bisect_left`).

In [2]:
import bisect

def primera_posicion(lista, objetivo):
    izq, der = 0, len(lista)
    while izq < der:
        medio = (izq + der) // 2
        if lista[medio] < objetivo:
            izq = medio + 1
        else:
            der = medio
    return izq

datos = [1, 2, 2, 2, 5, 8]
for x in (2, 3, 0, 9):
    assert primera_posicion(datos, x) == bisect.bisect_left(datos, x)
print(primera_posicion(datos, 2), primera_posicion(datos, 3))

1 4


## 3. Raíz cuadrada entera

Calcular $\lfloor\sqrt{n}\rfloor$ buscando binariamente el mayor $r$ con $r^2 \le n$.

In [3]:
import math

def raiz_entera(n):
    izq, der = 0, n
    while izq < der:
        # redondeo arriba evita bucle infinito
        medio = (izq + der + 1) // 2
        if medio * medio <= n:
            izq = medio
        else:
            der = medio - 1
    return izq

assert all(raiz_entera(n) == math.isqrt(n) for n in range(10_000))
print(raiz_entera(10**20 + 12345))

10000000000


## 4. `lower_bound` y `upper_bound`

Implementar `lower_bound` (primer índice con valor $\ge x$) y `upper_bound` (primer índice con valor $> x$), con un parámetro `clave` opcional, y compararlos con `bisect_left` y `bisect_right`. Usarlos para contar apariciones y extraer un rango de valores en $O(\log n)$.

In [4]:
import random

def identidad(x):
    return x

def lower_bound(lista, objetivo, clave=identidad):
    izq, der = 0, len(lista)
    while izq < der:
        medio = (izq + der) // 2
        if clave(lista[medio]) < objetivo:
            izq = medio + 1
        else:
            der = medio
    return izq

def upper_bound(lista, objetivo, clave=identidad):
    izq, der = 0, len(lista)
    while izq < der:
        medio = (izq + der) // 2
        # <= salta también los iguales
        if clave(lista[medio]) <= objetivo:
            izq = medio + 1
        else:
            der = medio
    return izq

random.seed(4)
for _ in range(500):
    datos = sorted(random.choices(range(20), k=random.randint(0, 20)))
    x = random.randint(-2, 22)
    assert lower_bound(datos, x) == bisect.bisect_left(datos, x)
    assert upper_bound(datos, x) == bisect.bisect_right(datos, x)
    assert upper_bound(datos, x) - lower_bound(datos, x) == datos.count(x)

datos = [1, 2, 2, 2, 5, 8, 9]
print(lower_bound(datos, 2), upper_bound(datos, 2), "→ el 2 aparece", upper_bound(datos, 2) - lower_bound(datos, 2), "veces")
print("Valores en [3, 8]:", datos[lower_bound(datos, 3):upper_bound(datos, 8)])

def edad(persona):
    return persona[1]

personas = [("Luis", 25), ("Pau", 25), ("Ana", 30), ("Eva", 30), ("Iker", 41)]
assert lower_bound(personas, 30, edad) == bisect.bisect_left(personas, 30, key=edad)
assert upper_bound(personas, 30, edad) == bisect.bisect_right(personas, 30, key=edad)
print("Con 30 años:", personas[lower_bound(personas, 30, edad):upper_bound(personas, 30, edad)])

1 4 → el 2 aparece 3 veces
Valores en [3, 8]: [5, 8]
Con 30 años: [('Ana', 30), ('Eva', 30)]


## 5. Búsqueda binaria sobre la respuesta

Si la respuesta $x$ cumple un predicado monótono (si $x$ vale, cualquier $x' > x$ también vale), se puede buscar binariamente en el rango de posibles respuestas. Escribir `primer_verdadero(izq, der, predicado)` y usarla para: a) la raíz entera del ejercicio 3; b) la capacidad mínima de un camión que reparte unos paquetes, en el orden dado, en como mucho $d$ días. Comprobar contra la búsqueda lineal de la respuesta.

In [5]:
def primer_verdadero(izq, der, predicado):
    # menor x en [izq, der] con predicado(x) cierto
    while izq < der:
        medio = (izq + der) // 2
        if predicado(medio):
            der = medio
        else:
            izq = medio + 1
    return izq

def raiz(n):
    return primer_verdadero(0, n + 1, lambda r: r * r > n) - 1

def dias_necesarios(pesos, capacidad):
    dias, carga = 1, 0
    for p in pesos:
        if carga + p > capacidad:
            dias, carga = dias + 1, 0
        carga += p
    return dias

def capacidad_minima(pesos, dias):
    return primer_verdadero(max(pesos), sum(pesos), lambda c: dias_necesarios(pesos, c) <= dias)

def capacidad_fuerza_bruta(pesos, dias):
    return next(c for c in range(max(pesos), sum(pesos) + 1) if dias_necesarios(pesos, c) <= dias)

assert all(raiz(n) == math.isqrt(n) for n in range(10_000))
random.seed(5)
for _ in range(300):
    pesos = random.choices(range(1, 15), k=random.randint(1, 12))
    d = random.randint(1, len(pesos))
    assert capacidad_minima(pesos, d) == capacidad_fuerza_bruta(pesos, d)

pesos = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
for d in (1, 3, 5, 10):
    print(f"{d:>2} días → capacidad {capacidad_minima(pesos, d)}")

 1 días → capacidad 55
 3 días → capacidad 21
 5 días → capacidad 15
10 días → capacidad 10


## 6. Lista ordenada y rotada

Una lista ordenada sin repetidos se ha rotado, por ejemplo `[15, 18, 22, 3, 5, 9, 12]`. Encontrar un valor en $O(\log n)$: en cada paso una de las dos mitades está ordenada y basta mirar si el objetivo cae en ella. Encontrar también el punto de giro (la posición del mínimo).

In [6]:
def busca_rotada(lista, objetivo):
    izq, der = 0, len(lista) - 1
    while izq <= der:
        medio = (izq + der) // 2
        if lista[medio] == objetivo:
            return medio
        if lista[izq] <= lista[medio]:  # mitad izquierda ordenada
            if lista[izq] <= objetivo < lista[medio]:
                der = medio - 1
            else:
                izq = medio + 1
        else:  # mitad derecha ordenada
            if lista[medio] < objetivo <= lista[der]:
                izq = medio + 1
            else:
                der = medio - 1
    return -1

def punto_de_giro(lista):
    izq, der = 0, len(lista) - 1
    while izq < der:
        medio = (izq + der) // 2
        if lista[medio] > lista[der]:
            izq = medio + 1
        else:
            der = medio
    return izq

rotada = [15, 18, 22, 3, 5, 9, 12]
print(busca_rotada(rotada, 5), busca_rotada(rotada, 18), busca_rotada(rotada, 7), punto_de_giro(rotada))

random.seed(6)
for _ in range(200):
    base = sorted(random.sample(range(100), random.randint(1, 15)))
    for k in range(len(base)):
        l = base[k:] + base[:k]
        assert punto_de_giro(l) == l.index(min(l))
        for x in base + [-1, 50.5, 100]:
            assert busca_rotada(l, x) == (l.index(x) if x in l else -1)

4 1 -1 3


## 7. Pico de una lista

Un pico es un elemento mayor que sus vecinos (fuera de la lista se considera $-\infty$). Si no hay dos elementos adyacentes iguales siempre existe uno; encontrarlo en $O(\log n)$ aunque la lista no esté ordenada: si `l[m] < l[m + 1]`, la lista sube y hay un pico a la derecha de $m$.

In [7]:
def pico(lista):
    izq, der = 0, len(lista) - 1
    while izq < der:
        medio = (izq + der) // 2
        if lista[medio] < lista[medio + 1]:
            izq = medio + 1
        else:
            der = medio
    return izq

def es_pico(lista, i):
    return (i == 0 or lista[i - 1] < lista[i]) and (i == len(lista) - 1 or lista[i + 1] < lista[i])

datos = [1, 3, 20, 4, 1, 0, 7, 2]
print(pico(datos), [i for i in range(len(datos)) if es_pico(datos, i)])

random.seed(7)
for _ in range(1000):
    l = random.sample(range(100), random.randint(1, 30))
    assert es_pico(l, pico(l))

2 [2, 6]
